# 🧑‍🍳 Neosantara AI Cookbook: RAG with ChromaDB

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neosantara-xyz/examples/blob/main/cookbook/advanced/rag-chromadb.ipynb)

This recipe demonstrates how to build a complete Retrieval-Augmented Generation (RAG) pipeline using Neosantara AI for embeddings and chat completions, and ChromaDB as the vector store.

In [ ]:
# Install dependencies
!pip install chromadb openai

# Setup API Key (for Google Colab)
try:
    from google.colab import userdata
    import os
    os.environ["NEOSANTARA_API_KEY"] = userdata.get('NEOSANTARA_API_KEY')
    print("API Key loaded from Google Colab userdata.")
except ImportError:
    import os
    if "NEOSANTARA_API_KEY" in os.environ:
        print("API Key loaded from environment variables.")
    else:
        print("Not running in Google Colab and NEOSANTARA_API_KEY not found in environment.")

In [ ]:
from openai import OpenAI
import os
import chromadb
from chromadb.utils import embedding_functions

# Initialize Neosantara Client
client = OpenAI(
    api_key=os.getenv("NEOSANTARA_API_KEY"),
    base_url="https://api.neosantara.xyz/v1"
)

## 1. Embeddings with Neosantara AI

We use the `nusa-embedding-0001` model to generate vector representations of our text data.

In [ ]:
def get_embeddings(text):
    response = client.embeddings.create(
        input=text,
        model="nusa-embedding-0001"
    )
    return response.data[0].embedding

# Example embedding
embedding = get_embeddings("Neosantara AI is a powerful platform for Indonesian language models.")
print(f"Embedding dimension: {len(embedding)}")

## 2. Vector Store Setup (ChromaDB)

We'll create an in-memory ChromaDB collection and add some sample knowledge.

In [ ]:
# Initialize ChromaDB
chroma_client = chromadb.Client()

# Create a custom embedding function for ChromaDB
class NeosantaraEmbeddingFunction(embedding_functions.EmbeddingFunction):
    def __call__(self, input):
        embeddings = []
        for text in input:
            embeddings.append(get_embeddings(text))
        return embeddings

# Create collection
collection = chroma_client.create_collection(
    name="knowledge_base",
    embedding_function=NeosantaraEmbeddingFunction()
)

# Add sample documents
documents = [
    "Neosantara AI was founded in 2024 to provide state-of-the-art AI models for the Nusantara region.",
    "The platform supports a variety of tasks including text generation, vision, and embeddings.",
    "Nusa-embedding-0001 is optimized for Indonesian and regional languages.",
    "Neosantara AI's mission is to democratize access to advanced AI for Indonesian developers."
]
ids = ["doc1", "doc2", "doc3", "doc4"]

collection.add(
    documents=documents,
    ids=ids
)

print("Documents added to ChromaDB.")

## 3. Retrieval & Generation (RAG)

Now we'll query the vector store for relevant context and use it to generate a grounded answer.

In [ ]:
query = "What is the mission of Neosantara AI?"

# 1. Retrieve relevant context from ChromaDB
results = collection.query(
    query_texts=[query],
    n_results=2
)
context = " ".join(results['documents'][0])

print(f"Retrieved Context: {context}\n")

# 2. Generate response using the context
response = client.chat.completions.create(
    model="nusantara-base",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Use the following context to answer the user's question."},
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {query}"}
    ]
)

print(f"Generated Answer: {response.choices[0].message.content}")